# CS781 Phase 1 — Model 1 (GQA + RoPE) — Calibration & Training Run

Run this once with `STEPS_THIS_RUN` small (default 1) to measure seconds/step. The timing cell then tells you how many steps fit in your remaining 30-hour budget. Commit, add this notebook's own output as an input dataset on the next run, bump `STEPS_THIS_RUN` up, and re-run — it resumes automatically from the last checkpoint.

Cells 1, 2, 4, 5, 6, 7 (structure), 8, and the whole generation/submission section at the bottom are unchanged from the course-provided starter notebook, apart from one line in cell 6 (an `<eos>` token inserted between documents) and swapping GPT-2 for the custom architecture in cell 9.

## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

if torch.cuda.device_count() > 1:
    print()
    print("NOTE: 2 GPUs visible. Running Trainer directly in a notebook cell")
    print("(no torchrun/accelerate launcher) means it will use nn.DataParallel,")
    print("not true DistributedDataParallel. That's fine for this calibration")
    print("run -- the timing you get back is the real, honest number for")
    print("however this actually executes. Switching to true DDP later is a")
    print("separate, optional speed optimization on top of this.")

## 3. Configuration

#### Architecture is fixed from the design phase. The only thing you change run-to-run is `STEPS_THIS_RUN`.

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import shutil

from transformers.trainer_utils import get_last_checkpoint

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Start with one file to validate the pipeline; switch to looping over all
# four Acts/High-Courts/Supreme-Court/Tribunals files once this runs clean.
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# --- Model architecture (locked from design phase) ---
MODEL_CONTEXT_LENGTH = 1024
TRAIN_BLOCK_SIZE = 1024
N_EMBD = 1024
N_LAYER = 8
N_HEAD = 16
N_KV_HEAD = 4
D_FF = 2688

# --- Data ---
# 120M-param model 1 is Chinchilla-optimal around ~2.4B tokens, but your
# T4x2/30hr compute budget (~5.6B tokens for this size) covers the entire
# ~4.5B-token corpus with room to spare -- so don't cap short of it, use it
# all. Set comfortably above the real corpus size so the break-early logic
# in cell 7 never triggers.
TRAIN_TOKENS = 5_000_000_000
VAL_TOKENS = 5_000_000
SEED = 42
random.seed(SEED)

# Fraction of documents to keep from EACH of the 4 files (applied in cell 5).
# The OOM crash was really the token-list memory bug fixed in cell 7/8 below,
# not document count -- but starting at 0.5 right after a crash gives extra
# headroom for anything else in the pipeline. Once a run completes cleanly,
# push this back toward 1.0 to use the full corpus.
DOC_SAMPLE_FRACTION = 0.5

# BPE vocab size for the from-scratch tokenizer (single-domain legal corpus,
# smaller than GPT-2's 50257 is reasonable -- adjust if your merges look
# too coarse/fine once trained).
VOCAB_SIZE = 32_000

# --- THIS is what you change every run ---
# First run: leave at 1 just to confirm nothing crashes and get a (noisy,
# overestimated) per-step time. Second run: bump to ~20-50 for a stable
# timing read. After that: set it to whatever the timing cell suggests for
# your remaining budget, and just keep re-running to consume it.
STEPS_THIS_RUN = 1

# Checkpoint frequency: independent of STEPS_THIS_RUN, so you always have a
# resumable checkpoint even if a long run gets killed by Kaggle's ~12hr wall
# limit before finishing. Lower this once you know your real step time if
# 250 steps takes too long between saves.
SAVE_STEPS = 250

# Conservative starting point -- if this run doesn't OOM, raise
# PER_DEVICE_BATCH before your next real (non-calibration) run.
PER_DEVICE_BATCH = 8
GRAD_ACCUM_STEPS = 1

TOTAL_GPU_HOUR_BUDGET = 30.0

# --- Generation (Task 1) ---
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 256
TEMPERATURE = 0.7
TOP_P = 0.95

# --- Kaggle output ---
OUTPUT_DIR = "/kaggle/working/cs781_phase1_model1"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-gqa-final")
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "tokenizer")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
TRAINING_META_PATH = os.path.join(OUTPUT_DIR, "training_meta.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration loaded. STEPS_THIS_RUN =", STEPS_THIS_RUN)

## 4. Download Training Data From Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

train_file_paths = [
    hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
    for fname in TRAIN_FILES
]

print("Training files:")
for p in train_file_paths:
    print(" ", p)

## 5. Load Training Records

#### Streams each file with `ijson` instead of `json.load()`. `json.load()` requires materializing the ENTIRE parsed file in memory before anything can be discarded -- your high_courts file alone is 10.7GB raw, which inflates to ~20-25GB once parsed into Python objects, and that's what actually caused the crash (before `DOC_SAMPLE_FRACTION` ever got a chance to discard anything). Streaming applies the sampling decision per-document as the file is read, so peak memory scales with what you *keep*, not the file's full size. Because it's a per-document random draw rather than an exact-count sample, the kept count per file will be close to but not exactly `DOC_SAMPLE_FRACTION` of the total. Then shuffles the combined result (seeded, reproducible) so the `TRAIN_TOKENS` cap in cell 7 can't starve any one file.

In [ ]:
!pip install -q ijson

In [ ]:
import ijson

random.seed(SEED)

records = []
for path, fname in zip(train_file_paths, TRAIN_FILES):
    kept, seen = 0, 0
    with open(path, "rb") as f:
        for rec in ijson.items(f, "item"):
            seen += 1
            if random.random() < DOC_SAMPLE_FRACTION:
                records.append(rec)
                kept += 1
    print(f"{fname}: kept {kept:,} of {seen:,} documents")

random.shuffle(records)
print(f"Total documents (shuffled): {len(records):,}")

## 6. Train BPE Tokenizer From Scratch

#### Project rules prohibit a pretrained tokenizer, so this trains its own byte-level BPE on the legal corpus. It fingerprints which files it was trained on (`TRAIN_FILES`) and only reuses a saved tokenizer if that fingerprint still matches -- so switching from one file to all four correctly triggers a retrain instead of silently reusing a tokenizer that only ever saw Supreme Court text.

In [ ]:
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast, AutoTokenizer

TOKENIZER_META_PATH = os.path.join(TOKENIZER_DIR, "trained_on.json")
current_fingerprint = sorted(TRAIN_FILES)

existing_fingerprint = None
if os.path.exists(TOKENIZER_META_PATH):
    with open(TOKENIZER_META_PATH) as f:
        existing_fingerprint = json.load(f).get("train_files")

if os.path.isdir(TOKENIZER_DIR) and existing_fingerprint == current_fingerprint:
    print("Existing tokenizer matches current TRAIN_FILES -- loading it (not retraining).")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
else:
    if existing_fingerprint is not None:
        print("TRAIN_FILES changed since the saved tokenizer was trained -- retraining.")
    else:
        print("No existing tokenizer found -- training a new one from scratch.")

    def text_iterator():
        for rec in records:
            yield rec["text"]

    bpe = ByteLevelBPETokenizer()
    bpe.train_from_iterator(
        text_iterator(),
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=["<|endoftext|>"],
    )

    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=bpe._tokenizer,
        bos_token="<|endoftext|>",
        eos_token="<|endoftext|>",
        unk_token="<|endoftext|>",
        pad_token="<|endoftext|>",
    )
    os.makedirs(TOKENIZER_DIR, exist_ok=True)
    tokenizer.save_pretrained(TOKENIZER_DIR)
    with open(TOKENIZER_META_PATH, "w") as f:
        json.dump({"train_files": current_fingerprint}, f)

tokenizer.padding_side = "left"  # required for correct batched generation later
print("Vocabulary size:", len(tokenizer))

## 7. Create Train and Validation Tokens Set

#### Reuses `records` from cell 5. Two changes from the starter: an `<eos>` between each document's tokens (unrelated legal texts being concatenated), and token IDs are accumulated as `numpy.uint16` arrays rather than a plain Python list. A Python list of billions of ints costs ~28 bytes/int in overhead alone -- that's what actually triggered the memory-restart crash, not the number of documents. uint16 (2 bytes, covers vocab sizes up to 65535) cuts that by ~14x.

In [ ]:
import numpy as np

token_dtype = np.uint16 if VOCAB_SIZE < 65536 else np.uint32
train_chunks, val_chunks = [], []
train_count, val_count = 0, 0

random.seed(SEED)

for data in records:
    text = data["text"]
    ids = tokenizer.encode(text, add_special_tokens=False)
    ids.append(tokenizer.eos_token_id)
    arr = np.array(ids, dtype=token_dtype)

    if random.random() < 0.05:
        if val_count < VAL_TOKENS:
            arr = arr[: VAL_TOKENS - val_count]
            val_chunks.append(arr)
            val_count += len(arr)
    else:
        if train_count < TRAIN_TOKENS:
            arr = arr[: TRAIN_TOKENS - train_count]
            train_chunks.append(arr)
            train_count += len(arr)

    if train_count >= TRAIN_TOKENS and val_count >= VAL_TOKENS:
        break

train_tokens = np.concatenate(train_chunks) if train_chunks else np.array([], dtype=token_dtype)
val_tokens = np.concatenate(val_chunks) if val_chunks else np.array([], dtype=token_dtype)

print(f"Train tokens: {len(train_tokens):,}  ({train_tokens.nbytes/1e9:.2f} GB)")
print(f"Validation tokens: {len(val_tokens):,}  ({val_tokens.nbytes/1e9:.2f} GB)")

## 8. Create Token Blocks

In [ ]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    return tokens[:total_length].reshape(-1, block_size)

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)

print("Training sequences:", train_blocks.shape[0])
print("Validation sequences:", val_blocks.shape[0])

## 9. Create Datasets

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})

print(train_dataset)
print(val_dataset)

## 10. Define The Model

#### Custom GQA + RoPE + SwiGLU + RMSNorm decoder, wrapped as a HF-compatible `PreTrainedModel` so `Trainer`, checkpointing, and `save_pretrained`/`from_pretrained` all work exactly as they would with `GPT2LMHeadModel`.

Written to an actual file and imported (rather than defined directly in this cell) because recent `transformers` versions read `__file__` off a model class's module during init -- a class defined live in a notebook cell lives in `__main__`, which has no `__file__` in a Jupyter/papermill kernel, and that lookup crashes with an `AttributeError` before training even starts.

In [ ]:
%%writefile /kaggle/working/legal_gqa_model.py
"""
Model 1 for CS781 Phase-1: GQA + RoPE + SwiGLU + RMSNorm decoder-only transformer.

Wrapped as a transformers PretrainedConfig/PreTrainedModel subclass so it is a
drop-in replacement for GPT2Config/GPT2LMHeadModel in the starter notebook:
same save_pretrained/from_pretrained, same Trainer + TrainingArguments +
resume_from_checkpoint behavior, same model.generate() call signature.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast


class LegalGQAConfig(PretrainedConfig):
    model_type = "legal_gqa"

    def __init__(
        self,
        vocab_size=50257,
        n_positions=1024,
        n_embd=1024,
        n_layer=8,
        n_head=16,
        n_kv_head=4,
        d_ff=2688,
        rope_theta=10000.0,
        dropout=0.1,
        initializer_range=0.02,
        rms_norm_eps=1e-5,
        bos_token_id=50256,
        eos_token_id=50256,
        pad_token_id=50256,
        **kwargs,
    ):
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.vocab_size = vocab_size
        self.n_positions = n_positions
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.d_ff = d_ff
        self.rope_theta = rope_theta
        self.dropout = dropout
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = True
        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    # q, k: (B, n_head, T, head_dim); cos/sin: (T, head_dim)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


class GroupedQueryAttention(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.n_rep = self.n_head // self.n_kv_head
        self.dropout_p = config.dropout

        self.q_proj = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_head * self.head_dim, config.n_embd, bias=False)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        present = (k, v) if use_cache else None

        # GQA: broadcast each KV head across its group of query heads
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_bias,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        return self.o_proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.up_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.down_proj = nn.Linear(config.d_ff, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.ffn = SwiGLU(config)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        attn_out, present = self.attn(
            self.ln_1(x), cos, sin, attn_bias, is_causal, past_key_value, use_cache
        )
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, present


class LegalGQAForCausalLM(PreTrainedModel):
    config_class = LegalGQAConfig

    def __init__(self, config: LegalGQAConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

        # precompute RoPE tables once up to the max context length
        inv_freq = 1.0 / (
            config.rope_theta
            ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim)
        )
        t = torch.arange(config.n_positions).float()
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("rope_cos", emb.cos(), persistent=False)
        self.register_buffer("rope_sin", emb.sin(), persistent=False)

        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        # scaled init on residual-stream output projections (GPT-2 recipe)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(
                    p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer)
                )

    def get_input_embeddings(self):
        return self.wte

    def set_input_embeddings(self, value):
        self.wte = value

    def _rope_slice(self, past_len, T, device, dtype):
        cos = self.rope_cos[past_len : past_len + T].to(device=device, dtype=dtype)
        sin = self.rope_sin[past_len : past_len + T].to(device=device, dtype=dtype)
        return cos, sin

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        """Returns (attn_bias, is_causal). attn_bias is None when the fast
        SDPA is_causal fused-kernel path is usable (no padding to mask)."""
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1

        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min

        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None, :, :]
        return bias, False

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        use_cache=None,
        **kwargs,
    ):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(
            attention_mask, past_len, T, input_ids.device, x.dtype
        )

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            x, present = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
            if use_cache:
                presents.append(present)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return CausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

    # No prepare_inputs_for_generation / GenerationMixin here on purpose —
    # generation is done by generate_utils.generate_batch(), which calls
    # this model's forward() directly with its own tuple-based KV cache
    # rather than going through model.generate().

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")
from legal_gqa_model import LegalGQAConfig, LegalGQAForCausalLM

config = LegalGQAConfig(
    vocab_size=len(tokenizer),
    n_positions=MODEL_CONTEXT_LENGTH,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    d_ff=D_FF,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)
model = LegalGQAForCausalLM(config)

model_size = sum(p.numel() for p in model.parameters())
print(f"Model size: {model_size / 1e6:.2f}M parameters")

## 10b. Load Generation Utilities

#### Manual sampling loop (no `model.generate()`) with top-p+temperature sampling and two repetition guards: blocks a 3rd consecutive repeat of the same word, and blocks recurrence of any trigram already generated. Written to a file for the same `__main__.__file__` reason as cell 10.

In [ ]:
%%writefile /kaggle/working/generate_utils.py
"""
Drop-in replacement for the starter notebook's generate_batch() in
'## 19. Generate 128 Words'. Same signature and return shape, so cell 20
(the prediction loop) and cells 21-22 (prediction.json / submission.zip)
need no changes at all.

Written as a plain per-step loop against the model's own forward() and its
tuple-based KV cache (see legal_gqa_model.py) rather than model.generate(),
to avoid depending on transformers' internal Cache-class machinery.
"""
import torch
import torch.nn.functional as F


def _top_p_filter(logits, top_p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits, dim=-1)
    cum_probs = torch.cumsum(probs, dim=-1)
    # keep the smallest prefix whose cumulative prob exceeds top_p;
    # always keep at least the top-1 token
    remove = cum_probs > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out.scatter_(-1, sorted_idx, sorted_logits)
    return out


def _build_word_token_table(tokenizer):
    """token_id -> lowercased word it produces, only for tokens that start a
    new word (GPT-2 byte-level BPE marks these with a leading space)."""
    table = {}
    for tid in range(len(tokenizer)):
        text = tokenizer.decode([tid])
        if text[:1] in (" ", "\n"):
            word = text.strip().lower()
            if word:
                table.setdefault(word, []).append(tid)
    return {w: torch.tensor(ids) for w, ids in table.items()}


@torch.inference_mode()
def generate_batch(
    prompts,
    model,
    tokenizer,
    device,
    target_words=128,
    max_new_tokens=384,
    temperature=0.7,
    top_p=0.95,
    tail_window=8,  # tokens looked at when checking for a repeated word
):
    model.eval()
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True,
        max_length=model.config.n_positions - max_new_tokens,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    input_lengths = attention_mask.sum(dim=1)
    B = input_ids.size(0)

    word_table = getattr(generate_batch, "_word_table_cache", None)
    if word_table is None:
        word_table = _build_word_token_table(tokenizer)
        generate_batch._word_table_cache = word_table  # cache across calls

    # prefill: run the full prompt once to seed the KV cache
    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True)
    past_key_values = out.past_key_values
    next_logits = out.logits[:, -1, :]

    generated = [[] for _ in range(B)]
    trigram_seen = [set() for _ in range(B)]
    finished_word_count = [0] * B
    cur_ids = input_ids
    cur_mask = attention_mask

    for _ in range(max_new_tokens):
        step_logits = next_logits / max(temperature, 1e-5)

        for row in range(B):
            if finished_word_count[row] >= target_words:
                continue

            # --- guard 1: block a 3rd consecutive repeat of the same word ---
            tail_ids = torch.tensor(generated[row][-tail_window:])
            if tail_ids.numel() > 0:
                tail_text = tokenizer.decode(tail_ids, skip_special_tokens=True)
                words = tail_text.split()
                if len(words) >= 2 and words[-1].lower() == words[-2].lower():
                    block_ids = word_table.get(words[-1].lower())
                    if block_ids is not None:
                        step_logits[row, block_ids.to(device)] = float("-inf")

        # --- guard 2: block recurrence of any trigram already generated ---
        # checked only against each row's current top-50 candidates (the
        # tokens sampling would realistically pick anyway) rather than the
        # full vocab, since this runs every generation step.
        for row in range(B):
            if finished_word_count[row] >= target_words or len(generated[row]) < 2:
                continue
            prefix = (generated[row][-2], generated[row][-1])
            seen = trigram_seen[row]
            # only tokens that would recreate a seen trigram get blocked;
            # we look this up lazily per-row since the seen set is small
            for tid in torch.topk(step_logits[row], k=min(50, step_logits.size(-1))).indices.tolist():
                if (prefix[0], prefix[1], tid) in seen:
                    step_logits[row, tid] = float("-inf")

        filtered = _top_p_filter(step_logits, top_p)
        probs = F.softmax(filtered, dim=-1)
        next_tokens = torch.multinomial(probs, num_samples=1).squeeze(-1)

        for row in range(B):
            if finished_word_count[row] >= target_words:
                next_tokens[row] = tokenizer.eos_token_id
                continue
            tok = next_tokens[row].item()
            generated[row].append(tok)
            if len(generated[row]) >= 3:
                trigram_seen[row].add(tuple(generated[row][-3:]))
            text_so_far = tokenizer.decode(generated[row], skip_special_tokens=True)
            finished_word_count[row] = len(text_so_far.split())

        if all(c >= target_words for c in finished_word_count):
            break

        next_col = next_tokens.unsqueeze(1)
        cur_mask = torch.cat([cur_mask, torch.ones(B, 1, dtype=cur_mask.dtype, device=device)], dim=1)
        out = model(input_ids=next_col, attention_mask=cur_mask, past_key_values=past_key_values, use_cache=True)
        past_key_values = out.past_key_values
        next_logits = out.logits[:, -1, :]

    predictions = []
    for row in range(B):
        text = tokenizer.decode(generated[row], skip_special_tokens=True)
        words = text.split()[:target_words]
        predictions.append(" ".join(words))
    return predictions

In [ ]:
from generate_utils import generate_batch

In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 11. Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    max_steps=None,  # set per-run below, after we know how many steps already exist
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=2000,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),  # T4 is Turing: fp16, not bf16
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    logging_steps=max(1, SAVE_STEPS // 10),
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
)

## 12. Training (with resume + per-step timing)

#### This is the cell that answers \"how long does one step take, and how many more fit in my budget\". It resumes automatically if a checkpoint already exists in `MODEL_OUTPUT_DIR`.

In [ ]:
from transformers import Trainer

last_checkpoint = get_last_checkpoint(MODEL_OUTPUT_DIR) if os.path.isdir(MODEL_OUTPUT_DIR) else None
steps_at_start = int(last_checkpoint.split("-")[-1]) if last_checkpoint else 0
print(f"Resuming from step {steps_at_start}" if last_checkpoint else "Starting fresh (no checkpoint found).")

training_args.max_steps = steps_at_start + STEPS_THIS_RUN

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

start_time = time.time()
trainer.train(resume_from_checkpoint=last_checkpoint)
elapsed = time.time() - start_time

steps_this_session = trainer.state.global_step - steps_at_start
sec_per_step = elapsed / max(steps_this_session, 1)

print()
print(f"Steps completed this run: {steps_this_session}")
print(f"Wall time this run: {elapsed/3600:.3f} hours ({elapsed:.1f}s)")
print(f"Seconds/step: {sec_per_step:.2f}")
if steps_this_session <= 5:
    print("NOTE: with this few steps, the first step's one-off cuDNN/kernel")
    print("autotuning overhead is inflating this number. Treat it as a safe")
    print("upper bound, not the steady-state rate -- rerun with")
    print("STEPS_THIS_RUN ~20-50 for a tighter estimate.")

# persist cumulative GPU time across sessions (this file must travel with the
# checkpoint when you re-attach last session's output as this session's input)
cumulative_seconds = 0.0
if os.path.exists(TRAINING_META_PATH):
    with open(TRAINING_META_PATH) as f:
        cumulative_seconds = json.load(f).get("cumulative_seconds", 0.0)
cumulative_seconds += elapsed
with open(TRAINING_META_PATH, "w") as f:
    json.dump({"cumulative_seconds": cumulative_seconds, "global_step": trainer.state.global_step}, f)

budget_seconds = TOTAL_GPU_HOUR_BUDGET * 3600
remaining_seconds = max(budget_seconds - cumulative_seconds, 0)
remaining_steps = int(remaining_seconds / sec_per_step) if sec_per_step > 0 else 0

print()
print(f"Cumulative GPU time used: {cumulative_seconds/3600:.2f} / {TOTAL_GPU_HOUR_BUDGET:.0f} hours")
print(f"At this rate, remaining budget fits ~{remaining_steps:,} more steps")
print(f"For your next run: set STEPS_THIS_RUN so that")
print(f"  steps_at_start + STEPS_THIS_RUN <= {trainer.state.global_step + remaining_steps:,}")

## 13. Perplexity

In [ ]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")

## 14. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps, train_loss, eval_steps, eval_loss_list = [], [], [], []
for log in logs:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if "eval_loss" in log and "step" in log:
        eval_steps.append(log["step"])
        eval_loss_list.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.plot(eval_steps, eval_loss_list, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Phase 1 Model 1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:")
print(LOSS_PLOT_PATH)

## 15. Save Final Model to Kaggle Output

#### Also commit this notebook so `MODEL_OUTPUT_DIR` (checkpoints + `training_meta.json`) survives as this version's output -- that's what you attach as an input dataset to resume next session.

In [ ]:
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("Final model saved to:")
print(MODEL_OUTPUT_DIR)

## 16. Load Test Data from HuggingFace

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
generation_tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_DIR)
generation_model = LegalGQAForCausalLM.from_pretrained(MODEL_OUTPUT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

## 19. Generate Predictions for the Test Set

#### Unchanged from the starter apart from calling our `generate_batch` -- same function name and signature, so nothing else here needed to move.

In [ ]:
predictions = {}
GENERATION_BATCH_SIZE = 32

items = list(test_data.items())
total_examples = len(items)
print(f"Total test examples: {total_examples}")

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_ids = [str(example_id) for example_id, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    batch_predictions = generate_batch(
        prompts=batch_prompts,
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=TARGET_GENERATION_WORDS,
        max_new_tokens=MAX_GENERATION_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )

    for example_id, prediction in zip(batch_ids, batch_predictions):
        predictions[example_id] = {"text": prediction}

    completed = start + len(batch)
    print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 20. Save prediction.json

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("prediction.json saved:")
print(PREDICTION_PATH)

## 21. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)